# 32 - Humanoid Mechatronics And Actuators

## What / Why / How

**What we are trying to do:** Study actuator sizing, torque-speed tradeoffs, power, thermal load, and series elasticity.

**Why this matters:** Humanoid capability is limited by actuators and energy as much as by AI. Weak or hot actuators make impressive demos impossible.

**How we will do it:** Estimate joint torque and power for simple motions, compare gear ratios, and simulate a compliant actuator response.

## Actuator Sizing

Human-scale robots live or die by torque, speed, mass, heat, and cost. This notebook keeps the math simple but forces the right engineering questions.

In [ ]:
import math
import random
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

try:
    import matplotlib.pyplot as plt
    HAS_PLOT = True
except ModuleNotFoundError:
    plt = None
    HAS_PLOT = False

np.set_printoptions(precision=3, suppress=True)

def plot_unavailable():
    if not HAS_PLOT:
        print("Install matplotlib to see plots: pip install -r requirements.txt")

In [ ]:
gravity = 9.81
segment_mass = 4.0
segment_length = 0.45
payload = 2.0
lever_arm = 0.35
safety_factor = 2.0

shoulder_torque = safety_factor * gravity * (segment_mass * segment_length / 2 + payload * lever_arm)
print("estimated shoulder hold torque [Nm]:", round(shoulder_torque, 2))

In [ ]:
motor_speed = np.linspace(0, 6000, 80)  # rpm
stall_torque = 0.8  # Nm at motor
no_load_speed = 6000
motor_torque = stall_torque * np.maximum(0, 1 - motor_speed / no_load_speed)
gear_ratios = [30, 60, 100]
for ratio in gear_ratios:
    joint_speed = motor_speed / ratio
    joint_torque = motor_torque * ratio * 0.75
    print(f"ratio={ratio:3}: max joint torque={joint_torque.max():.1f} Nm, max joint speed={joint_speed.max():.1f} rpm")

In [ ]:
from robotics_mastery.humanoid import actuator_power

torque = np.array([10, 20, 30, 20, 10], dtype=float)
velocity = np.array([0.5, 1.0, 1.2, -0.8, -0.4])
power = actuator_power(torque, velocity)
print("instantaneous mechanical power [W]:", power)
print("positive power mean [W]:", power[power > 0].mean())

In [ ]:
stiffness = 120.0
damping = 2.5
load_torque = 8.0
motor_angle = 0.2
joint_angle, joint_velocity = 0.0, 0.0
rows = []
for step in range(300):
    spring_torque = stiffness * (motor_angle - joint_angle)
    net_torque = spring_torque - damping * joint_velocity - load_torque
    joint_velocity += net_torque * 0.002
    joint_angle += joint_velocity * 0.002
    rows.append((step * 0.002, joint_angle, spring_torque))
rows = np.array(rows)
print("final joint angle:", rows[-1, 1])
print("final spring torque:", rows[-1, 2])

In [ ]:
if HAS_PLOT:
    plt.figure(figsize=(7, 3))
    plt.plot(rows[:, 0], rows[:, 1], label="joint angle")
    plt.plot(rows[:, 0], rows[:, 2] / stiffness, label="spring deflection equivalent")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.title("Compliant actuator toy response")
    plt.show()
else:
    plot_unavailable()

## Exercises

1. Estimate knee torque for standing from a squat.
2. Add motor temperature as a state.
3. Explain the tradeoff between high gear ratio and backdrivability.